In [1]:
#!pip install adversarial-robustness-toolbox
#!pip install tensorflow
#import sys
#!{sys.executable} -m pip install torch

In [2]:
import numpy as np
import tensorflow as tf
from keras.datasets import mnist#cifar10
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from art.attacks.evasion import SaliencyMapMethod
from art.estimators.classification  import KerasClassifier
from sklearn.metrics import accuracy_score

In [3]:
tf.compat.v1.disable_eager_execution()

In [4]:
import tensorflow as tf
import json

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)
x_train = x_train.astype("float32")
x_test = x_test.astype("float32")
x_train /= 255
x_test /= 255
# download mnist data and split into train and test sets
#(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
# reshape data to fit model
X_train, X_test = x_train/255, x_test/255
# one-hot encode target column
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
# create model
model = Sequential()
model.add(Conv2D(32, (3, 3), padding='same', input_shape=(28, 28, 1), activation='relu'))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

# Create a neural network model
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=x_train.shape[1:]))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))

In [5]:
# compile model using accuracy as a measure of model performance
model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=['accuracy'])

# train model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5)

json.dump({'model': model.to_json()}, open("model.json", "w"))
model.save_weights("model_weights.h5")


Train on 60000 samples, validate on 10000 samples
Epoch 1/5
59968/60000 [============================>.] - ETA: 0s - loss: 0.6528 - accuracy: 0.7878

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2333: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates


60000/60000 [==============================] - 142s 2ms/sample - loss: 0.6525 - accuracy: 0.7879 - val_loss: 0.2733 - val_accuracy: 0.9179
Epoch 2/5
60000/60000 [==============================] - 131s 2ms/sample - loss: 0.2296 - accuracy: 0.9302 - val_loss: 0.1665 - val_accuracy: 0.9504
Epoch 3/5
60000/60000 [==============================] - 134s 2ms/sample - loss: 0.1493 - accuracy: 0.9550 - val_loss: 0.1149 - val_accuracy: 0.9648
Epoch 4/5
60000/60000 [==============================] - 132s 2ms/sample - loss: 0.1127 - accuracy: 0.9657 - val_loss: 0.0993 - val_accuracy: 0.9695
Epoch 5/5
60000/60000 [==============================] - 134s 2ms/sample - loss: 0.0920 - accuracy: 0.9718 - val_loss: 0.1127 - val_accuracy: 0.9634


In [6]:
#model.fit(x_train_adv, y_train_adv, batch_size=32, epochs=10, validation_data=(x_test, y_test))
# Evaluate the model's accuracy on the test dataset
score1 = model.evaluate(X_test, y_test, verbose=0)
print('Test accuracy:', score1[1])
#score1 = model.evaluate(x_test_adv, y_test_adv, verbose=0)
#print('Test accuracy:', score1[1])

Test accuracy: 0.9634


In [7]:
#Create a KerasClassifier for the model
classifier = KerasClassifier(model=model, clip_values=(0, 1),  use_logits=False)
# Generate adversarial examples
x_test = X_test # your test data
y = y_test # the true labels for the test data
jsma = SaliencyMapMethod(classifier=classifier, theta=1, gamma=0.1)
#x_test_adv = jsma.generate(x=x_test, y=y)
x_test_adv = jsma.generate(x_test)

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2357: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


JSMA:   0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
x_test_adv
x_test
print(f"X_test shape: {x_test_adv.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
model.fit(x_train, y_train, batch_size=32, epochs=5, validation_data=(x_test_adv, y_test))
# Evaluate the model's accuracy on the test dataset
#score1 = model.evaluate(x_test, y_test, verbose=0)
#print('Test accuracy:', score1[1])
score1 = model.evaluate(x_test_adv, y_test, verbose=0)
print('Test accuracy:', score1[1])

In [ ]:
import matplotlib.pyplot as plt
img_num = 590
img = x_test[img_num]
img1 = x_test_adv[img_num]
# Show the image
plt.imshow(img)

plt.show()

preds_adv = model.predict(x_test_adv)

#preds = np.argmax(model.predict(x_test_adv), axis=1)
#confidence = np.max(model.predict(x_test_adv), axis=1)
# Get the class label and confidence score
class_label = np.argmax(preds_adv, axis=1)
confidence = np.max(preds_adv, axis=1)

print("Class Label_adv: ", class_label[img_num])
print("Confidence_adv: ", confidence[img_num])

preds = model.predict(x_test)

# Get the class label and confidence score
class_label = np.argmax(preds, axis=1)
confidence = np.max(preds, axis=1)

print("Class Label: ", class_label[img_num])
print("Confidence: ", confidence[img_num])

In [ ]:
plt.imshow(img1)
plt.show()

In [ ]:
#### With adversrial confirdence
'''prob = model.predict(x_test_adv)
preds = np.argmax(model.predict(x_test_adv), axis=1)
acc = np.sum(preds == np.argmax(y_test, axis=1)) / y_test.shape[0]
print('Test accuracy on adversarial examples: {}'.format(acc))
print('Test confidence on adversarial examples: {}'.format(prob))'''

In [ ]:
preds = np.argmax(model.predict(X_test), axis=1)
acc = np.sum(preds == np.argmax(y_test, axis=1)) / y_test.shape[0]
acc

In [ ]:
preds = np.argmax(model.predict(X_train), axis=1)
acc = np.sum(preds == np.argmax(y_train, axis=1)) / y_train.shape[0]
acc

In [ ]:
#x_adv_combine=np.concatenate((X_train, x_test_adv), axis=0)

In [ ]:
#y_final=np.concatenate((y_train, y_test), axis=0)

In [ ]:
'''print(f"X_test shape: {x_adv_combine.shape}")
print(f"y_test shape: {y_train.shape}")
x_test_adv
print(f"y_test shape: {x_test_adv.shape}")'''


In [ ]:
#model.fit(x_adv_combine, y_final, validation_data=(X_test, y_test), epochs=5)


In [ ]:
#model.fit(X_train, y_train, validation_data=(x_test_adv, y_test), epochs=5)


In [ ]:
#y_pred_original = model.predict(X_test)
#y_pred_original_cat = tf.keras.utils.to_categorical(y_pred_original)

#y_test

In [ ]:
#preds_adv = model.predict(x_test_adv)
#y_test_adv = tf.keras.utils.to_categorical(preds_adv)


In [ ]:
#y_pred= model.predict(X_test)

In [ ]:
#y_pred

In [ ]:
import tensorflow as tf

# Load a pre-trained model
#model = tf.keras.applications.MobileNetV2()

img_num = x_train.shape
img = x_train[img_num]
img1 = x_train_adv[img_num]

# Load an image and preprocess it for the model
img = tf.keras.preprocessing.image.load_img("image.jpg", target_size=(224, 224))
img = tf.keras.preprocessing.image.img_to_array(img)
img = tf.keras.applications.mobilenet_v2.preprocess_input(img)

# Perform the prediction
predictions = model.predict(img)

# Get the class label and confidence score
label = tf.keras.applications.mobilenet_v2.decode_predictions(predictions, top=1)
class_label = label[0][0][1]
confidence = label[0][0][2]

print("Class Label: ", class_label)
print("Confidence: ", confidence)
In this example, the image is preprocessed before being passed to the model. The model then returns a set of predictions, which are decoded to get the class label and confidence score.
Note that the decode_predictions function will return the class label and confidence score for the top-n predictions, in this case, the top 1.